In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
#loading data
df_txn=pd.read_csv('./transactions.csv')
df_prod=pd.read_csv('./product-lookup.csv')
df_cust=pd.read_csv('./customers.csv')

In [3]:
#initial checks - shape, head, info, describe
df_txn.shape
df_txn.head(10)

,transaction_id,Transaction Date,account_number,Amount,txn_type,Merchant Name,status,customer_age
0,TXN00001,2022-02-02,ACC1015,312.65,withdrawal,BIGBASKET,REVERSED,57
1,TXN00002,06/04/2022,ACC1025,120.61,credit,paytm,FAILED,31
2,TXN00003,"August 27, 2022",ACC1045,392.11,dbit,NETFLIX,SUCCESS,58
3,TXN00004,2023-09-09,ACC1010,1457.6,withdrawal,BigBasket,REVERSED,30
4,TXN00005,29/07/2023,ACC1046,104.46,debit,bigbasket,PENDING,45
5,TXN00006,"October 28, 2023",ACC1014,104.46,WITHDRAWAL,Uber,FAILED,56
6,TXN00007,06/03/2023,ACC1005,1585.76,Transfer,flipkart,PENDING,56
7,TXN00008,"April 05, 2023",ACC1027,469.26,deposit,Uber,PENDING,57
8,TXN00009,2022-10-12,ACC1027,73.39,CREDIT,Flipkart,SUCCESS,39
9,TXN00010,13 Jun 2022,ACC1022,334.9,debit,PhonePe,FAILED,60


In [4]:
#df_txn.info()
df_txn.describe(include='all')

,transaction_id,Transaction Date,account_number,Amount,txn_type,Merchant Name,status,customer_age
count,305,305,305,294,305,305,284,305.000000
unique,290,285,75,287,29,49,4,NaN
top,TXN00225,02-13-2023,ACC1045,73.12,withdrawal,HDFC ATM,FAILED,NaN
freq,2,3,11,2,28,14,76,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.691803
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,56.538675
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.000000
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42.000000
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53.000000


In [5]:
df_txn.dtypes

transaction_id      object
Transaction Date    object
account_number      object
Amount              object
txn_type            object
Merchant Name       object
status              object
customer_age         int64
dtype: object

In [6]:
#data cleaning - #1 column names
df=df_txn.copy()
print("BEFORE:", list(df.columns))
df.columns = (df.columns
              .str.strip()          # remove leading/trailing spaces
              .str.lower()          # lowercase
              .str.replace(' ','_') # spaces → underscores
             )
print("AFTER: ", list(df.columns))

BEFORE: ['transaction_id', 'Transaction Date', 'account_number', 'Amount', 'txn_type', 'Merchant Name', 'status', 'customer_age']
AFTER:  ['transaction_id', 'transaction_date', 'account_number', 'amount', 'txn_type', 'merchant_name', 'status', 'customer_age']


In [7]:
#data cleaning - #2 Remove Duplicates
print(f"Rows before: {len(df)}")
print(f"Duplicate transaction_ids: {df['transaction_id'].duplicated().sum()}")
print(f"Fully duplicate rows: {df.duplicated().sum()}")
#Remove fully duplicate rows first
df = df.drop_duplicates()
print(f"\nAfter drop_duplicates(): {len(df)} rows")
#Keep first occurrence of duplicate transaction_ids
df = df.drop_duplicates(subset=['transaction_id'], keep='first')
print(f"After dedup on transaction_id: {len(df)} rows")

Rows before: 305
Duplicate transaction_ids: 15
Fully duplicate rows: 5

After drop_duplicates(): 300 rows
After dedup on transaction_id: 290 rows


In [8]:
#data cleaning - #3: Fix Date Formats
print("Sample dates (before):")
print(df['transaction_date'].dropna().sample(10, random_state=1).tolist())
print(f"\nCurrent dtype: {df['transaction_date'].dtype}")
#df['transaction_date'] = df['transaction_date'].str.strip() 
df['transaction_date'] = pd.to_datetime(
    df['transaction_date'],
    dayfirst=True,          # handles DD/MM/YYYY
    format='mixed',
    errors='coerce'         # invalid → NaT instead of crashing
)
nat_count = df['transaction_date'].isna().sum()
print(f"\nAfter pd.to_datetime:")
print(f"  dtype: {df['transaction_date'].dtype}")
print(f"  NaT (unparseable): {nat_count}")
print("Sample dates (after):", df['transaction_date'].dropna().sample(5, random_state=1).tolist())


Sample dates (before):
['08/06/2022', '2023-11-03', '08 Jan 2023', '22/03/2023', '2022-02-25', '06-16-2022', '03-07-2022', '09/12/2023', '08/12/2023', 'July 02, 2022']

Current dtype: object

After pd.to_datetime:
  dtype: datetime64[ns]
  NaT (unparseable): 0
Sample dates (after): [Timestamp('2022-06-08 00:00:00'), Timestamp('2023-11-03 00:00:00'), Timestamp('2023-01-08 00:00:00'), Timestamp('2023-03-22 00:00:00'), Timestamp('2022-02-25 00:00:00')]


In [9]:
#data cleaning - #4: Clean account_number
print("Sample before:", df['account_number'].dropna().sample(8, random_state=1).tolist())
mask=df['account_number'].str.islower()
print("Sample odds ones:",df.loc[mask,['account_number']].sample(8, random_state=1))
df['account_number'] = (df['account_number']
                        .str.strip()   # remove spaces
                        .str.upper()   # force uppercase: ACC1001
                       )


Sample before: ['ACC1020', 'ACC1012', 'ACC1048', 'ACC1004', 'ACC1046', 'ACC1049', 'ACC1013', 'ACC1012']
Sample odds ones:     account_number
37         acc1032
77         acc1049
76         acc1018
33         acc1048
205        acc1036
43         acc1025
27         acc1020
288        acc1041


In [10]:
#data cleaning - #5: Fix Amount Column
print("Sample before:", df['amount'].dropna().sample(10, random_state=1).tolist())
print(f"dtype before: {df['amount'].dtype}")
#remove $ signs
df['amount'] = df['amount'].str.replace('$', '', regex=False)
#Strip whitespace
df['amount'] = df['amount'].str.strip()
#Replace 'nan' string back to actual NaN
df['amount'] = df['amount'].replace('nan', np.nan)
#Convert to float
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
#Handle negatives — flag as refunds, take absolute value
refund_mask = df['amount'] < 0
df['is_refund'] = refund_mask.astype(int)
df['amount'] = df['amount'].abs()

Sample before: ['$73.8', '17.37', '72.79', '530.81', '296.83', '172.34', '235.32', '17.84', '8.42', '161.95']
dtype before: object


In [11]:
#data cleaning - #6 standardize txn type & clean names
print("Unique values before:")
print(df['txn_type'].value_counts())
df['txn_type'] = df['txn_type'].str.strip().str.lower()

# Fix typos using a mapping
typo_map = {
    'crdit': 'credit', 'crdit ': 'credit',
    'dbit':  'debit',  'dbit ':  'debit',
    'trnsfr':'transfer','dpst': 'deposit','dposit': 'deposit',
    'wthdrwl':'withdrawal',
}
df['txn_type'] = df['txn_type'].replace(typo_map)
print("\nUnique values after:")
print(df['txn_type'].value_counts().to_string())

#Clean Merchant Name
print("Sample before:", df['merchant_name'].dropna().unique()[:8].tolist())

df['merchant_name'] = df['merchant_name'].str.strip().str.title()

print("Sample after: ", df['merchant_name'].dropna().unique()[:8].tolist())

Unique values before:
txn_type
withdrawal     25
debit          15
withdrawal     15
deposit        14
debit          14
credit         13
transfr        12
Transfer       11
credit         11
dposit         10
Credit         10
 debit         10
 transfer      10
Deposit        10
DEBIT          10
deposit        10
WITHDRAWAL     10
transfer        9
dbit            9
TRANSFER        9
DEPOSIT         7
Debit           7
CREDIT          7
crdit           7
Withdrawal      6
 credit         6
 withdrawal     5
 deposit        5
transfer        3
Name: count, dtype: int64

Unique values after:
txn_type
debit         65
withdrawal    61
deposit       56
credit        54
transfer      42
transfr       12
Sample before: ['BIGBASKET', 'paytm', 'NETFLIX', 'BigBasket  ', 'bigbasket', '  Uber', 'flipkart', 'Uber  ']
Sample after:  ['Bigbasket', 'Paytm', 'Netflix', 'Uber', 'Flipkart', 'Phonepe', 'Amazon', 'Zomato']


In [12]:
#data cleaning - #7 #handle missing values
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])
# Different strategies for different columns
# status: categorical → fill with 'UNKNOWN'
df['status'] = df['status'].fillna('UNKNOWN')
print(f"\nstatus: filled with UNKNOWN")

# amount: numerical → fill with median (robust to outliers)
amt_median = df['amount'].median()
df['amount'] = df['amount'].fillna(amt_median)
print(f"amount: filled with median ({amt_median:.2f})")

# transaction_date: drop rows where date is missing
before = len(df)
df = df.dropna(subset=['transaction_date'])
print(f"transaction_date: dropped {before - len(df)} rows with NaT")

Missing values per column:
amount    11
status    19
dtype: int64

status: filled with UNKNOWN
amount: filled with median (164.49)
transaction_date: dropped 0 rows with NaT


In [13]:
#data cleaning - #8 handle outlier values
print(f"customer_age stats before:\n{df['customer_age'].describe()}")
print(f"\nOutlier values: {sorted(df[df['customer_age'] < 18]['customer_age'].unique().tolist() + df[df['customer_age'] > 100]['customer_age'].unique().tolist())}")

# Strategy: replace outliers with NaN, then fill with median
valid_mask = (df['customer_age'] >= 18) & (df['customer_age'] <= 100)
outlier_count = (~valid_mask).sum()

df.loc[~valid_mask, 'customer_age'] = np.nan
age_median = df['customer_age'].median()
df['customer_age'] = df['customer_age'].fillna(age_median)

customer_age stats before:
count    290.000000
mean      45.831034
std       57.912376
min        0.000000
25%       32.000000
50%       42.000000
75%       53.000000
max      999.000000
Name: customer_age, dtype: float64

Outlier values: [0, 150, 999]


In [14]:
#data prep for merging
print("\ntransactions.csv — account_number:")
print(f"  dtype:   {df['account_number'].dtype}")
print(f"  nulls:   {df['account_number'].isna().sum()}")
print(f"  sample:  {df['account_number'].dropna().unique()[:5].tolist()}")

print("\ncustomers.csv — account_number:")
print(f"  dtype:   {df_cust['account_number'].dtype}")
print(f"  nulls:   {df_cust['account_number'].isna().sum()}")
print(f"  sample:  {df_cust['account_number'].unique()[:5].tolist()}")

# Check overlap
txn_accounts  = set(df['account_number'].dropna().unique())
cust_accounts = set(df_cust['account_number'].unique())

in_both      = txn_accounts & cust_accounts
only_in_txn  = txn_accounts - cust_accounts
only_in_cust = cust_accounts - txn_accounts

print(f"\nAccounts in BOTH:          {len(in_both)}")
print(f"Only in transactions:      {len(only_in_txn)} ← will be LOST in inner join!")
print(f"Only in customers:         {len(only_in_cust)} ← no transactions for these")


transactions.csv — account_number:
  dtype:   object
  nulls:   0
  sample:  ['ACC1015', 'ACC1025', 'ACC1045', 'ACC1010', 'ACC1046']

customers.csv — account_number:
  dtype:   object
  nulls:   0
  sample:  ['ACC1015', 'ACC1019', 'ACC1018', 'ACC1012', 'ACC1037']

Accounts in BOTH:          45
Only in transactions:      5 ← will be LOST in inner join!
Only in customers:         0 ← no transactions for these


In [16]:
#merging Prepare product_lookup key ────────────────────
print("merc/kjhgfdsahant_name in transactions (sample):", df['merchant_name'].unique()[:5].tolist())
print("merchant_name in product_lookup:        ", df_prod['merchant_name'].unique().tolist())

# Already cleaned merchant_name above — check if they match now
matches = df['merchant_name'].isin(df_prod['merchant_name'])
print(f"\nTransaction rows with matching merchant: {matches.sum()} / {len(df)}")
print(f"No match: {(~matches).sum()} rows (merchants not in lookup)")

merc/kjhgfdsahant_name in transactions (sample): ['Bigbasket', 'Paytm', 'Netflix', 'Uber', 'Flipkart']
merchant_name in product_lookup:         ['Amazon', 'Netflix', 'Uber', 'Swiggy', 'Zomato', 'BigBasket', 'Flipkart', 'PhonePe', 'Paytm', 'HDFC ATM']

Transaction rows with matching merchant: 197 / 290
No match: 93 rows (merchants not in lookup)


In [17]:
# Add year-month column for analysis ────────────
df['year_month'] = df['transaction_date'].dt.to_period('M').astype(str)
df['day_of_week'] = df['transaction_date'].dt.day_name()
print("Added: year_month, day_of_week")
print(df[['transaction_date','year_month','day_of_week']].head(5))

# Sort datasets ──────────────────────────────────
df = df.sort_values('transaction_date').reset_index(drop=True)
print(f"\nDataset sorted by transaction_date")
print(f"Date range: {df['transaction_date'].min()} to {df['transaction_date'].max()}")

Added: year_month, day_of_week
  transaction_date year_month day_of_week
0       2022-02-02    2022-02   Wednesday
1       2022-04-06    2022-04   Wednesday
2       2022-08-27    2022-08    Saturday
3       2023-09-09    2023-09    Saturday
4       2023-07-29    2023-07    Saturday

Dataset sorted by transaction_date
Date range: 2022-01-02 00:00:00 to 2024-01-01 00:00:00


In [18]:
#Primary Merge — transactions + customers ───────
df_merged = pd.merge(
    df,           # left: all transactions
    df_cust,      # right: customer data
    on='account_number',
    how='left'    # keep all transactions
)

print(f"transactions rows: {len(df)}")
print(f"customers rows:    {len(df_cust)}")
print(f"merged rows:       {len(df_merged)}")
print(f"new columns added: {[c for c in df_merged.columns if c not in df.columns]}")

#Second Merge — add product categories ──────────
df_final = pd.merge(
    df_merged,
    df_prod,
    left_on='merchant_name',
    right_on='merchant_name',
    how='left'
)

print(f"After product_lookup merge: {df_final.shape}")
print(f"New columns: {[c for c in df_final.columns if c not in df_merged.columns]}")

transactions rows: 290
customers rows:    45
merged rows:       290
new columns added: ['customer_name', 'city', 'join_date', 'credit_score', 'segment', 'email', 'phone']
After product_lookup merge: (290, 21)
New columns: ['category', 'is_digital', 'avg_txn_value']


In [19]:
#post merge validations
#Row count checks ──────────────────────────────
print(f"  Transactions (clean): {len(df)}")
print(f"  After +customers:     {len(df_merged)}")
print(f"  After +product_lookup:{len(df_final)}")

if len(df_final) == len(df):
    print("  ✓ Row count preserved (no accidental row multiplication)")
else:
    diff = len(df_final) - len(df)
    print(f"  ⚠ Row count changed by {diff} (investigate duplicates in join key!)")

#Check NaN rates after merge ───────────────────
null_pct = (df_final.isnull().sum() / len(df_final) * 100).round(1)
null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
print(null_pct.to_string())

#Duplicate check ───────────────────────────────
dup_after = df_final.duplicated(subset=['transaction_id']).sum()
print(f"Duplicate transaction_ids after merge: {dup_after}")
if dup_after > 0:
    print("⚠ WARNING: Duplicates created! Check if join key is 1-to-many")
else:
    print("✓ No duplicate transaction_ids (clean merge)")

#Sanity check aggregates ───────────────────────
print("Total transaction amount by status:")
print(df_final.groupby('status')['amount'].agg(['count','sum','mean']).round(2))

print("\nTransaction count by category:")
print(df_final['category'].value_counts())

print("\nTop 5 accounts by total spend:")
print(df_final.groupby('account_number')['amount']
      .sum().sort_values(ascending=False).head(5))

#Sample rows ───────────────────────────────────
key_cols = ['transaction_id','transaction_date','account_number',
            'customer_name','amount','txn_type','category','status']
available = [c for c in key_cols if c in df_final.columns]
print(df_final[available].sample(5, random_state=42).to_string())

  Transactions (clean): 290
  After +customers:     290
  After +product_lookup:290
  ✓ Row count preserved (no accidental row multiplication)
category         32.1
is_digital       32.1
avg_txn_value    32.1
credit_score     19.7
customer_name     9.0
city              9.0
join_date         9.0
segment           9.0
email             9.0
phone             9.0
Duplicate transaction_ids after merge: 0
✓ No duplicate transaction_ids (clean merge)
Total transaction amount by status:
          count       sum    mean
status                           
FAILED       72  27079.19  376.10
PENDING      59  31374.42  531.77
REVERSED     68  27311.78  401.64
SUCCESS      72  65759.02  913.32
UNKNOWN      19  11286.32  594.02

Transaction count by category:
category
Food Delivery    57
E-commerce       53
Payments         30
Entertainment    29
Transport        28
Name: count, dtype: int64

Top 5 accounts by total spend:
account_number
ACC1006    48300.00
ACC1021    12750.09
ACC1039     7880.42
ACC

In [20]:
# ── Save final clean dataset ─────────────────────────────────
df_final.to_csv('transactions_final_clean.csv', index=False)
print(f"\n✓ Saved: transactions_final_clean.csv ({df_final.shape})")


✓ Saved: transactions_final_clean.csv ((290, 21))


In [21]:
#Data required for LLM Prompt for Feature Engineering
print(df_final.head()) 
print(df_final.dtypes)

  transaction_id transaction_date account_number  amount txn_type  \
0       TXN00238       2022-01-02        ACC1023  196.31   credit   
1       TXN00135       2022-01-03        ACC1041   37.37   credit   
2       TXN00101       2022-01-08        ACC1016   17.76  deposit   
3       TXN00106       2022-01-09        ACC1037  272.07    debit   
4       TXN00177       2022-01-09        ACC1047  151.34  deposit   

  merchant_name    status  customer_age  is_refund year_month  ...  \
0        Swiggy   SUCCESS          48.0          0    2022-01  ...   
1      Hdfc Atm    FAILED          29.0          0    2022-01  ...   
2         Paytm    FAILED          62.0          0    2022-01  ...   
3     Bigbasket  REVERSED          49.0          0    2022-01  ...   
4        Swiggy   SUCCESS          48.0          0    2022-01  ...   

  customer_name       city   join_date credit_score  segment  \
0   Shweta Jain      Delhi  2019-12-02        612.0  Premium   
1    Kavita Rao       Pune  2022-03-